In [ ]:
# ============================================
# PROJECT 2: DEMAND & WORKLOAD FORECASTING
# NYC 311 SERVICE REQUESTS
# ============================================

import pandas as pd
import numpy as np
import requests
import sqlite3
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

PROJECT_FOLDER = Path("/content/project_2_demand_forecasting")
DATA_FOLDER = PROJECT_FOLDER / "data"
OUTPUT_FOLDER = PROJECT_FOLDER / "outputs"

DATA_FOLDER.mkdir(parents=True, exist_ok=True)
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

print("Project folders created successfully.")
print("Project folder:", PROJECT_FOLDER)

Project folders created successfully.
Project folder: /content/project_2_demand_forecasting


In [ ]:
# NYC Open Data dataset:
# 311 Service Requests from 2020 to Present

DATASET_ID = "erm2-nwe9"
API_URL = f"https://data.cityofnewyork.us/resource/{DATASET_ID}.json"

START_DATE = "2023-01-01T00:00:00"
END_DATE = "2026-07-01T00:00:00"  # Exclusive: includes through June 30, 2026


def fetch_nyc_open_data(query_parameters):
    """
    Retrieve aggregated NYC Open Data records.
    """

    response = requests.get(
        API_URL,
        params=query_parameters,
        timeout=180
    )

    print("HTTP status:", response.status_code)

    response.raise_for_status()

    data = response.json()
    dataframe = pd.DataFrame(data)

    if dataframe.empty:
        raise ValueError("The API returned no records.")

    return dataframe


print("API connection settings are ready.")

API connection settings are ready.


In [ ]:
daily_total_query = {
    "$select": """
        date_trunc_ymd(created_date) AS request_date,
        count(*) AS request_count
    """,
    "$where": f"""
        created_date >= '{START_DATE}'
        AND created_date < '{END_DATE}'
    """,
    "$group": "date_trunc_ymd(created_date)",
    "$order": "request_date ASC",
    "$limit": "5000"
}

daily_total = fetch_nyc_open_data(daily_total_query)

daily_total["request_date"] = pd.to_datetime(
    daily_total["request_date"],
    errors="coerce"
)

daily_total["request_count"] = pd.to_numeric(
    daily_total["request_count"],
    errors="coerce"
)

daily_total = (
    daily_total
    .dropna(subset=["request_date", "request_count"])
    .sort_values("request_date")
    .reset_index(drop=True)
)

print("Rows:", len(daily_total))
print("First date:", daily_total["request_date"].min())
print("Last date:", daily_total["request_date"].max())
print()
display(daily_total.head())
display(daily_total.tail())

HTTP status: 200
Rows: 1277
First date: 2023-01-01 00:00:00
Last date: 2026-06-30 00:00:00



,request_date,request_count
0,2023-01-01,6209
1,2023-01-02,6680
2,2023-01-03,8255
3,2023-01-04,8462
4,2023-01-05,8414


,request_date,request_count
1272,2026-06-26,11031
1273,2026-06-27,9885
1274,2026-06-28,10999
1275,2026-06-29,11949
1276,2026-06-30,11311


In [ ]:
segment_query = {
    "$select": """
        date_trunc_ymd(created_date) AS request_date,
        agency,
        borough,
        count(*) AS request_count
    """,
    "$where": f"""
        created_date >= '{START_DATE}'
        AND created_date < '{END_DATE}'
        AND borough IN (
            'BRONX',
            'BROOKLYN',
            'MANHATTAN',
            'QUEENS',
            'STATEN ISLAND'
        )
    """,
    "$group": """
        date_trunc_ymd(created_date),
        agency,
        borough
    """,
    "$order": """
        request_date ASC,
        agency ASC,
        borough ASC
    """,
    "$limit": "500000"
}

daily_segments = fetch_nyc_open_data(segment_query)

daily_segments["request_date"] = pd.to_datetime(
    daily_segments["request_date"],
    errors="coerce"
)

daily_segments["request_count"] = pd.to_numeric(
    daily_segments["request_count"],
    errors="coerce"
)

daily_segments["agency"] = (
    daily_segments["agency"]
    .astype(str)
    .str.strip()
    .str.upper()
)

daily_segments["borough"] = (
    daily_segments["borough"]
    .astype(str)
    .str.strip()
    .str.upper()
)

daily_segments = (
    daily_segments
    .dropna(
        subset=[
            "request_date",
            "agency",
            "borough",
            "request_count"
        ]
    )
    .sort_values(
        ["request_date", "borough", "agency"]
    )
    .reset_index(drop=True)
)

print("Segment rows:", len(daily_segments))
print("Agencies:", daily_segments["agency"].nunique())
print("Boroughs:", daily_segments["borough"].nunique())
print()
display(daily_segments.head(10))

HTTP status: 200
Segment rows: 76458
Agencies: 17
Boroughs: 5



,request_date,agency,borough,request_count
0,2023-01-01,DCWP,BRONX,4
1,2023-01-01,DEP,BRONX,21
2,2023-01-01,DHS,BRONX,4
3,2023-01-01,DOB,BRONX,20
4,2023-01-01,DOHMH,BRONX,12
5,2023-01-01,DOT,BRONX,22
6,2023-01-01,DPR,BRONX,18
7,2023-01-01,DSNY,BRONX,31
8,2023-01-01,HPD,BRONX,292
9,2023-01-01,NYPD,BRONX,777


In [ ]:
daily_total_file = DATA_FOLDER / "nyc_311_daily_total.csv"
daily_segments_file = DATA_FOLDER / "nyc_311_daily_segments.csv"

daily_total.to_csv(
    daily_total_file,
    index=False
)

daily_segments.to_csv(
    daily_segments_file,
    index=False
)

print("Files saved successfully:")
print(daily_total_file)
print(daily_segments_file)

Files saved successfully:
/content/project_2_demand_forecasting/data/nyc_311_daily_total.csv
/content/project_2_demand_forecasting/data/nyc_311_daily_segments.csv


In [ ]:
# ============================================
# STEP 6: DATA-QUALITY VALIDATION
# ============================================

print("DAILY TOTAL DATA QUALITY")
print("-" * 40)

print("Rows:", len(daily_total))
print("Duplicate dates:", daily_total["request_date"].duplicated().sum())
print("Missing dates:", daily_total["request_date"].isna().sum())
print("Missing request counts:", daily_total["request_count"].isna().sum())
print("Negative request counts:", (daily_total["request_count"] < 0).sum())

expected_dates = pd.date_range(
    start=daily_total["request_date"].min(),
    end=daily_total["request_date"].max(),
    freq="D"
)

missing_calendar_dates = expected_dates.difference(
    daily_total["request_date"]
)

print("Expected calendar days:", len(expected_dates))
print("Missing calendar days:", len(missing_calendar_dates))

print()
print("DAILY SEGMENT DATA QUALITY")
print("-" * 40)

print("Rows:", len(daily_segments))

segment_duplicates = daily_segments.duplicated(
    subset=["request_date", "agency", "borough"]
).sum()

print("Duplicate date-agency-borough rows:", segment_duplicates)
print("Missing dates:", daily_segments["request_date"].isna().sum())
print("Missing agencies:", daily_segments["agency"].isna().sum())
print("Missing boroughs:", daily_segments["borough"].isna().sum())
print("Missing request counts:", daily_segments["request_count"].isna().sum())
print("Negative request counts:", (daily_segments["request_count"] < 0).sum())

print()
print("REQUEST VOLUME SUMMARY")
print("-" * 40)

display(
    daily_total["request_count"]
    .describe()
    .to_frame("request_count")
)

print("Total service requests:", f"{daily_total['request_count'].sum():,.0f}")
print("Average daily requests:", f"{daily_total['request_count'].mean():,.2f}")
print("Highest daily requests:", f"{daily_total['request_count'].max():,.0f}")
print("Lowest daily requests:", f"{daily_total['request_count'].min():,.0f}")

highest_day = daily_total.loc[
    daily_total["request_count"].idxmax()
]

lowest_day = daily_total.loc[
    daily_total["request_count"].idxmin()
]

print()
print(
    "Highest-volume date:",
    highest_day["request_date"].date(),
    "-",
    f"{highest_day['request_count']:,.0f}"
)

print(
    "Lowest-volume date:",
    lowest_day["request_date"].date(),
    "-",
    f"{lowest_day['request_count']:,.0f}"
)

if (
    daily_total["request_date"].duplicated().sum() == 0
    and len(missing_calendar_dates) == 0
    and daily_total["request_count"].isna().sum() == 0
    and segment_duplicates == 0
):
    print()
    print("DATA-QUALITY RESULT: PASSED")
else:
    print()
    print("DATA-QUALITY RESULT: REVIEW REQUIRED")

DAILY TOTAL DATA QUALITY
----------------------------------------
Rows: 1277
Duplicate dates: 0
Missing dates: 0
Missing request counts: 0
Negative request counts: 0
Expected calendar days: 1277
Missing calendar days: 0

DAILY SEGMENT DATA QUALITY
----------------------------------------
Rows: 76458
Duplicate date-agency-borough rows: 0
Missing dates: 0
Missing agencies: 0
Missing boroughs: 0
Missing request counts: 0
Negative request counts: 0

REQUEST VOLUME SUMMARY
----------------------------------------


,request_count
count,"1,277.00"
mean,"9,656.31"
std,"1,583.18"
min,"5,596.00"
25%,"8,733.00"
50%,"9,538.00"
75%,"10,347.00"
max,"22,806.00"


Total service requests: 12,331,111
Average daily requests: 9,656.31
Highest daily requests: 22,806
Lowest daily requests: 5,596

Highest-volume date: 2026-02-24 - 22,806
Lowest-volume date: 2023-12-25 - 5,596

DATA-QUALITY RESULT: PASSED


In [ ]:
# ============================================
# STEP 7: CREATE SQLITE DATABASE
# ============================================

DATABASE_FILE = PROJECT_FOLDER / "nyc_311_forecasting.db"

connection = sqlite3.connect(DATABASE_FILE)

daily_total.to_sql(
    "daily_total",
    connection,
    if_exists="replace",
    index=False
)

daily_segments.to_sql(
    "daily_segments",
    connection,
    if_exists="replace",
    index=False
)

connection.execute("""
CREATE INDEX IF NOT EXISTS idx_daily_total_date
ON daily_total(request_date)
""")

connection.execute("""
CREATE INDEX IF NOT EXISTS idx_daily_segments_date
ON daily_segments(request_date)
""")

connection.execute("""
CREATE INDEX IF NOT EXISTS idx_daily_segments_borough
ON daily_segments(borough)
""")

connection.execute("""
CREATE INDEX IF NOT EXISTS idx_daily_segments_agency
ON daily_segments(agency)
""")

connection.commit()

table_check = pd.read_sql_query(
    """
    SELECT
        name AS table_name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
    """,
    connection
)

print("SQLite database created successfully.")
print("Database location:", DATABASE_FILE)
print()
display(table_check)

SQLite database created successfully.
Database location: /content/project_2_demand_forecasting/nyc_311_forecasting.db



,table_name
0,daily_segments
1,daily_total


In [ ]:
# ============================================
# STEP 8: VERIFY SQL TABLES
# ============================================

sql_verification = pd.read_sql_query(
    """
    SELECT
        'daily_total' AS table_name,
        COUNT(*) AS row_count,
        MIN(request_date) AS first_date,
        MAX(request_date) AS last_date
    FROM daily_total

    UNION ALL

    SELECT
        'daily_segments' AS table_name,
        COUNT(*) AS row_count,
        MIN(request_date) AS first_date,
        MAX(request_date) AS last_date
    FROM daily_segments
    """,
    connection
)

display(sql_verification)

,table_name,row_count,first_date,last_date
0,daily_total,1277,2023-01-01 00:00:00,2026-06-30 00:00:00
1,daily_segments,76458,2023-01-01 00:00:00,2026-06-30 00:00:00


In [ ]:
# ============================================
# STEP 9: SQL YEARLY WORKLOAD ANALYSIS
# ============================================

yearly_workload = pd.read_sql_query(
    """
    WITH yearly_summary AS (
        SELECT
            CAST(SUBSTR(request_date, 1, 4) AS INTEGER) AS request_year,
            SUM(request_count) AS total_requests,
            ROUND(AVG(request_count), 2) AS average_daily_requests,
            MAX(request_count) AS highest_daily_requests,
            MIN(request_count) AS lowest_daily_requests,
            COUNT(DISTINCT SUBSTR(request_date, 1, 10)) AS days_recorded
        FROM daily_total
        GROUP BY CAST(SUBSTR(request_date, 1, 4) AS INTEGER)
    ),

    yearly_growth AS (
        SELECT
            request_year,
            total_requests,
            average_daily_requests,
            highest_daily_requests,
            lowest_daily_requests,
            days_recorded,

            LAG(total_requests) OVER (
                ORDER BY request_year
            ) AS previous_year_requests
        FROM yearly_summary
    )

    SELECT
        request_year,
        total_requests,
        average_daily_requests,
        highest_daily_requests,
        lowest_daily_requests,
        days_recorded,
        previous_year_requests,

        CASE
            WHEN previous_year_requests IS NULL THEN NULL
            ELSE ROUND(
                (
                    total_requests - previous_year_requests
                ) * 100.0 / previous_year_requests,
                2
            )
        END AS year_over_year_growth_pct

    FROM yearly_growth
    ORDER BY request_year
    """,
    connection
)

print("YEARLY WORKLOAD AND GROWTH")
print("-" * 40)

display(yearly_workload)

YEARLY WORKLOAD AND GROWTH
----------------------------------------


,request_year,total_requests,average_daily_requests,highest_daily_requests,lowest_daily_requests,days_recorded,previous_year_requests,year_over_year_growth_pct
0,2023,3224723,"8,834.86",17962,5596,365,NaN,NaN
1,2024,3456770,"9,444.73",15899,6423,366,"3,224,723.00",7.20
2,2025,3655034,"10,013.79",16388,6216,365,"3,456,770.00",5.74
3,2026,1994584,"11,019.80",22806,7334,181,"3,655,034.00",-45.43


In [ ]:
# ============================================
# STEP 10: SQL YEAR-TO-DATE COMPARISON
# JANUARY 1 THROUGH JUNE 30
# ============================================

ytd_workload = pd.read_sql_query(
    """
    WITH ytd_summary AS (
        SELECT
            CAST(SUBSTR(request_date, 1, 4) AS INTEGER) AS request_year,
            SUM(request_count) AS total_requests,
            ROUND(AVG(request_count), 2) AS average_daily_requests,
            MAX(request_count) AS highest_daily_requests,
            MIN(request_count) AS lowest_daily_requests,
            COUNT(DISTINCT SUBSTR(request_date, 1, 10)) AS days_recorded
        FROM daily_total
        WHERE
            (
                request_date >= '2025-01-01'
                AND request_date < '2025-07-01'
            )
            OR
            (
                request_date >= '2026-01-01'
                AND request_date < '2026-07-01'
            )
        GROUP BY CAST(SUBSTR(request_date, 1, 4) AS INTEGER)
    ),

    ytd_growth AS (
        SELECT
            request_year,
            total_requests,
            average_daily_requests,
            highest_daily_requests,
            lowest_daily_requests,
            days_recorded,

            LAG(total_requests) OVER (
                ORDER BY request_year
            ) AS previous_ytd_requests

        FROM ytd_summary
    )

    SELECT
        request_year,
        total_requests,
        average_daily_requests,
        highest_daily_requests,
        lowest_daily_requests,
        days_recorded,
        previous_ytd_requests,

        CASE
            WHEN previous_ytd_requests IS NULL THEN NULL
            ELSE ROUND(
                (
                    total_requests - previous_ytd_requests
                ) * 100.0 / previous_ytd_requests,
                2
            )
        END AS ytd_growth_pct

    FROM ytd_growth
    ORDER BY request_year
    """,
    connection
)

print("JANUARY–JUNE YEAR-TO-DATE COMPARISON")
print("-" * 50)

display(ytd_workload)

JANUARY–JUNE YEAR-TO-DATE COMPARISON
--------------------------------------------------


,request_year,total_requests,average_daily_requests,highest_daily_requests,lowest_daily_requests,days_recorded,previous_ytd_requests,ytd_growth_pct
0,2025,1758816,"9,717.22",15023,7215,181,NaN,NaN
1,2026,1994584,"11,019.80",22806,7334,181,"1,758,816.00",13.40


In [ ]:
# ============================================
# STEP 11: SQL MONTHLY WORKLOAD ANALYSIS
# ============================================

monthly_workload = pd.read_sql_query(
    """
    WITH monthly_summary AS (
        SELECT
            SUBSTR(request_date, 1, 7) || '-01' AS month_start,
            CAST(SUBSTR(request_date, 1, 4) AS INTEGER) AS request_year,
            CAST(SUBSTR(request_date, 6, 2) AS INTEGER) AS request_month,
            SUM(request_count) AS total_requests,
            ROUND(AVG(request_count), 2) AS average_daily_requests,
            MAX(request_count) AS highest_daily_requests,
            MIN(request_count) AS lowest_daily_requests,
            COUNT(DISTINCT SUBSTR(request_date, 1, 10)) AS days_recorded
        FROM daily_total
        GROUP BY
            SUBSTR(request_date, 1, 7),
            CAST(SUBSTR(request_date, 1, 4) AS INTEGER),
            CAST(SUBSTR(request_date, 6, 2) AS INTEGER)
    ),

    monthly_comparison AS (
        SELECT
            *,
            LAG(total_requests, 1) OVER (
                ORDER BY month_start
            ) AS previous_month_requests,

            LAG(total_requests, 12) OVER (
                ORDER BY month_start
            ) AS previous_year_month_requests

        FROM monthly_summary
    )

    SELECT
        month_start,
        request_year,
        request_month,
        total_requests,
        average_daily_requests,
        highest_daily_requests,
        lowest_daily_requests,
        days_recorded,
        previous_month_requests,

        CASE
            WHEN previous_month_requests IS NULL THEN NULL
            ELSE ROUND(
                (
                    total_requests - previous_month_requests
                ) * 100.0 / previous_month_requests,
                2
            )
        END AS month_over_month_growth_pct,

        previous_year_month_requests,

        CASE
            WHEN previous_year_month_requests IS NULL THEN NULL
            ELSE ROUND(
                (
                    total_requests - previous_year_month_requests
                ) * 100.0 / previous_year_month_requests,
                2
            )
        END AS year_over_year_growth_pct

    FROM monthly_comparison
    ORDER BY month_start
    """,
    connection
)

monthly_workload["month_start"] = pd.to_datetime(
    monthly_workload["month_start"]
)

print("MONTHLY WORKLOAD ANALYSIS")
print("-" * 50)
print("Monthly records:", len(monthly_workload))
print()

display(monthly_workload.head(15))
display(monthly_workload.tail(12))

MONTHLY WORKLOAD ANALYSIS
--------------------------------------------------
Monthly records: 42



,month_start,request_year,request_month,total_requests,average_daily_requests,highest_daily_requests,lowest_daily_requests,days_recorded,previous_month_requests,month_over_month_growth_pct,previous_year_month_requests,year_over_year_growth_pct
0,2023-01-01,2023,1,242173,"7,812.03",9287,6209,31,NaN,NaN,NaN,NaN
1,2023-02-01,2023,2,225002,"8,035.79",10925,6588,28,"242,173.00",-7.09,NaN,NaN
2,2023-03-01,2023,3,251716,"8,119.87",10070,6535,31,"225,002.00",11.87,NaN,NaN
3,2023-04-01,2023,4,247496,"8,249.87",9567,6298,30,"251,716.00",-1.68,NaN,NaN
4,2023-05-01,2023,5,280932,"9,062.32",10959,7005,31,"247,496.00",13.51,NaN,NaN
5,2023-06-01,2023,6,267168,"8,905.60",10044,6892,30,"280,932.00",-4.90,NaN,NaN
6,2023-07-01,2023,7,289330,"9,333.23",11643,7665,31,"267,168.00",8.30,NaN,NaN
7,2023-08-01,2023,8,282757,"9,121.19",9991,8061,31,"289,330.00",-2.27,NaN,NaN
8,2023-09-01,2023,9,283517,"9,450.57",17962,6687,30,"282,757.00",0.27,NaN,NaN
9,2023-10-01,2023,10,302769,"9,766.74",13403,7015,31,"283,517.00",6.79,NaN,NaN


,month_start,request_year,request_month,total_requests,average_daily_requests,highest_daily_requests,lowest_daily_requests,days_recorded,previous_month_requests,month_over_month_growth_pct,previous_year_month_requests,year_over_year_growth_pct
30,2025-07-01,2025,7,315877,"10,189.58",11433,9447,31,"306,442.00",3.08,"298,936.00",5.67
31,2025-08-01,2025,8,304036,"9,807.61",10595,8658,31,"315,877.00",-3.75,"284,823.00",6.75
32,2025-09-01,2025,9,302684,"10,089.47",11395,8580,30,"304,036.00",-0.44,"306,770.00",-1.33
33,2025-10-01,2025,10,336612,"10,858.45",13908,8579,31,"302,684.00",11.21,"306,803.00",9.72
34,2025-11-01,2025,11,304905,"10,163.50",13453,8147,30,"336,612.00",-9.42,"293,516.00",3.88
35,2025-12-01,2025,12,332104,"10,713.03",16388,6216,31,"304,905.00",8.92,"313,949.00",5.78
36,2026-01-01,2026,1,348511,"11,242.29",19225,7334,31,"332,104.00",4.94,"348,180.00",0.10
37,2026-02-01,2026,2,334691,"11,953.25",22806,8888,28,"348,511.00",-3.97,"255,364.00",31.06
38,2026-03-01,2026,3,342386,"11,044.71",14146,9139,31,"334,691.00",2.30,"281,223.00",21.75
39,2026-04-01,2026,4,302189,"10,072.97",11962,7800,30,"342,386.00",-11.74,"272,550.00",10.87


In [ ]:
# ============================================
# STEP 12: SQL BOROUGH WORKLOAD ANALYSIS
# ============================================

borough_workload = pd.read_sql_query(
    """
    WITH daily_borough AS (
        SELECT
            SUBSTR(request_date, 1, 10) AS request_date,
            borough,
            SUM(request_count) AS daily_requests
        FROM daily_segments
        GROUP BY
            SUBSTR(request_date, 1, 10),
            borough
    ),

    borough_summary AS (
        SELECT
            borough,
            SUM(daily_requests) AS total_requests,
            ROUND(AVG(daily_requests), 2) AS average_daily_requests,
            MAX(daily_requests) AS highest_daily_requests,
            MIN(daily_requests) AS lowest_daily_requests,
            COUNT(DISTINCT request_date) AS days_recorded
        FROM daily_borough
        GROUP BY borough
    )

    SELECT
        borough,
        total_requests,
        average_daily_requests,
        highest_daily_requests,
        lowest_daily_requests,
        days_recorded,

        ROUND(
            total_requests * 100.0 /
            SUM(total_requests) OVER (),
            2
        ) AS workload_share_pct,

        DENSE_RANK() OVER (
            ORDER BY total_requests DESC
        ) AS workload_rank

    FROM borough_summary
    ORDER BY workload_rank
    """,
    connection
)

print("BOROUGH WORKLOAD ANALYSIS")
print("-" * 50)

display(borough_workload)

BOROUGH WORKLOAD ANALYSIS
--------------------------------------------------


,borough,total_requests,average_daily_requests,highest_daily_requests,lowest_daily_requests,days_recorded,workload_share_pct,workload_rank
0,BROOKLYN,3741882,"2,930.21",7001,1635,1277,30.38,1
1,QUEENS,2979442,"2,333.16",5108,1457,1277,24.19,2
2,BRONX,2604648,"2,039.66",7109,878,1277,21.15,3
3,MANHATTAN,2519723,"1,973.16",3625,959,1277,20.46,4
4,STATEN ISLAND,472137,369.72,4698,155,1277,3.83,5


In [ ]:
# ============================================
# STEP 13: SQL AGENCY WORKLOAD ANALYSIS
# ============================================

agency_workload = pd.read_sql_query(
    """
    WITH daily_agency AS (
        SELECT
            SUBSTR(request_date, 1, 10) AS request_date,
            agency,
            SUM(request_count) AS daily_requests
        FROM daily_segments
        GROUP BY
            SUBSTR(request_date, 1, 10),
            agency
    ),

    agency_summary AS (
        SELECT
            agency,
            SUM(daily_requests) AS total_requests,
            ROUND(AVG(daily_requests), 2) AS average_daily_requests,
            MAX(daily_requests) AS highest_daily_requests,
            MIN(daily_requests) AS lowest_daily_requests,
            COUNT(DISTINCT request_date) AS active_days
        FROM daily_agency
        GROUP BY agency
    )

    SELECT
        agency,
        total_requests,
        average_daily_requests,
        highest_daily_requests,
        lowest_daily_requests,
        active_days,

        ROUND(
            total_requests * 100.0 /
            SUM(total_requests) OVER (),
            2
        ) AS workload_share_pct,

        DENSE_RANK() OVER (
            ORDER BY total_requests DESC
        ) AS workload_rank

    FROM agency_summary
    ORDER BY workload_rank
    """,
    connection
)

print("AGENCY WORKLOAD ANALYSIS")
print("-" * 50)
print("Agencies analyzed:", len(agency_workload))
print()

display(agency_workload)

AGENCY WORKLOAD ANALYSIS
--------------------------------------------------
Agencies analyzed: 17



,agency,total_requests,average_daily_requests,highest_daily_requests,lowest_daily_requests,active_days,workload_share_pct,workload_rank
0,NYPD,5488112,"4,297.66",11306,2164,1277,44.55,1
1,HPD,2666246,"2,096.11",8228,438,1272,21.65,2
2,DSNY,1130916,885.60,11736,230,1277,9.18,3
3,DOT,769299,602.43,2905,138,1277,6.25,4
4,DEP,664466,520.33,4789,121,1277,5.39,5
5,DPR,440124,344.65,2002,28,1277,3.57,6
6,DOB,364482,285.42,2675,13,1277,2.96,7
7,DOHMH,299403,234.46,410,67,1277,2.43,8
8,DHS,174354,137.29,275,1,1270,1.42,9
9,TLC,126928,99.40,223,25,1277,1.03,10


In [ ]:
# ============================================
# STEP 14: SQL DAY-OF-WEEK SEASONALITY
# ============================================

weekday_workload = pd.read_sql_query(
    """
    WITH weekday_summary AS (
        SELECT
            CAST(STRFTIME('%w', request_date) AS INTEGER) AS weekday_number,

            CASE CAST(STRFTIME('%w', request_date) AS INTEGER)
                WHEN 0 THEN 'Sunday'
                WHEN 1 THEN 'Monday'
                WHEN 2 THEN 'Tuesday'
                WHEN 3 THEN 'Wednesday'
                WHEN 4 THEN 'Thursday'
                WHEN 5 THEN 'Friday'
                WHEN 6 THEN 'Saturday'
            END AS weekday_name,

            SUM(request_count) AS total_requests,
            ROUND(AVG(request_count), 2) AS average_daily_requests,
            MAX(request_count) AS highest_daily_requests,
            MIN(request_count) AS lowest_daily_requests,
            COUNT(*) AS days_recorded

        FROM daily_total

        GROUP BY
            CAST(STRFTIME('%w', request_date) AS INTEGER)
    )

    SELECT
        weekday_number,
        weekday_name,
        total_requests,
        average_daily_requests,
        highest_daily_requests,
        lowest_daily_requests,
        days_recorded,

        ROUND(
            total_requests * 100.0 /
            SUM(total_requests) OVER (),
            2
        ) AS workload_share_pct,

        DENSE_RANK() OVER (
            ORDER BY average_daily_requests DESC
        ) AS average_workload_rank

    FROM weekday_summary
    ORDER BY weekday_number
    """,
    connection
)

print("DAY-OF-WEEK WORKLOAD ANALYSIS")
print("-" * 50)

display(weekday_workload)

DAY-OF-WEEK WORKLOAD ANALYSIS
--------------------------------------------------


,weekday_number,weekday_name,total_requests,average_daily_requests,highest_daily_requests,lowest_daily_requests,days_recorded,workload_share_pct,average_workload_rank
0,0,Sunday,1631909,"8,917.54",15684,6209,183,13.23,6
1,1,Monday,1873447,"10,237.42",16388,5596,183,15.19,2
2,2,Tuesday,1881843,"10,283.30",22806,7373,183,15.26,1
3,3,Wednesday,1801709,"9,899.50",16982,6461,182,14.61,3
4,4,Thursday,1772932,"9,741.38",15361,6216,182,14.38,5
5,5,Friday,1778553,"9,772.27",17962,7340,182,14.42,4
6,6,Saturday,1590718,"8,740.21",13022,6423,182,12.90,7


In [ ]:
# ============================================
# STEP 15: FORECAST FEATURE ENGINEERING
# ============================================

# Start with a fresh copy of the validated daily dataset
forecast_data = daily_total.copy()

forecast_data = (
    forecast_data
    .sort_values("request_date")
    .reset_index(drop=True)
)

# --------------------------------------------
# 1. CALENDAR AND TREND FEATURES
# --------------------------------------------

forecast_data["trend_index"] = (
    forecast_data["request_date"]
    - forecast_data["request_date"].min()
).dt.days

forecast_data["year"] = forecast_data["request_date"].dt.year
forecast_data["month"] = forecast_data["request_date"].dt.month
forecast_data["weekday"] = forecast_data["request_date"].dt.dayofweek
forecast_data["day_of_year"] = forecast_data["request_date"].dt.dayofyear

forecast_data["is_weekend"] = (
    forecast_data["weekday"] >= 5
).astype(int)

# --------------------------------------------
# 2. CYCLICAL SEASONAL FEATURES
# --------------------------------------------

forecast_data["annual_sin"] = np.sin(
    2 * np.pi
    * forecast_data["day_of_year"]
    / 365.25
)

forecast_data["annual_cos"] = np.cos(
    2 * np.pi
    * forecast_data["day_of_year"]
    / 365.25
)

# --------------------------------------------
# 3. HISTORICAL LAG FEATURES
# --------------------------------------------

forecast_data["lag_1"] = (
    forecast_data["request_count"]
    .shift(1)
)

forecast_data["lag_7"] = (
    forecast_data["request_count"]
    .shift(7)
)

forecast_data["lag_14"] = (
    forecast_data["request_count"]
    .shift(14)
)

forecast_data["lag_28"] = (
    forecast_data["request_count"]
    .shift(28)
)

# --------------------------------------------
# 4. ROLLING STATISTICAL FEATURES
# Uses previous observations only
# --------------------------------------------

forecast_data["rolling_mean_7"] = (
    forecast_data["request_count"]
    .shift(1)
    .rolling(window=7)
    .mean()
)

forecast_data["rolling_mean_28"] = (
    forecast_data["request_count"]
    .shift(1)
    .rolling(window=28)
    .mean()
)

forecast_data["rolling_std_28"] = (
    forecast_data["request_count"]
    .shift(1)
    .rolling(window=28)
    .std()
)

# --------------------------------------------
# 5. CREATE WEEKDAY AND MONTH DUMMY VARIABLES
# --------------------------------------------

forecast_data = pd.get_dummies(
    forecast_data,
    columns=["weekday", "month"],
    prefix=["weekday", "month"],
    dtype=int
)

# --------------------------------------------
# 6. REMOVE INITIAL INCOMPLETE LAG ROWS
# --------------------------------------------

forecast_model_data = (
    forecast_data
    .dropna()
    .reset_index(drop=True)
)

# --------------------------------------------
# 7. DEFINE MODEL FEATURE COLUMNS
# --------------------------------------------

model_feature_columns = [
    "trend_index",
    "year",
    "day_of_year",
    "is_weekend",
    "annual_sin",
    "annual_cos",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "rolling_std_28"
]

dummy_feature_columns = [
    column
    for column in forecast_model_data.columns
    if column.startswith("weekday_")
    or column.startswith("month_")
]

model_feature_columns += dummy_feature_columns

# --------------------------------------------
# 8. CREATE 90-DAY HOLDOUT TEST PERIOD
# --------------------------------------------

test_start_date = (
    forecast_model_data["request_date"].max()
    - pd.Timedelta(days=89)
)

training_data = forecast_model_data[
    forecast_model_data["request_date"] < test_start_date
].copy()

testing_data = forecast_model_data[
    forecast_model_data["request_date"] >= test_start_date
].copy()

# --------------------------------------------
# 9. VALIDATE RESULTS
# --------------------------------------------

missing_model_values = forecast_model_data[
    model_feature_columns + ["request_count"]
].isna().sum().sum()

print("FORECAST DATA PREPARATION")
print("-" * 50)

print("Complete model rows:", len(forecast_model_data))
print("Model features:", len(model_feature_columns))

print()
print("Training rows:", len(training_data))
print(
    "Training period:",
    training_data["request_date"].min().date(),
    "through",
    training_data["request_date"].max().date()
)

print()
print("Testing rows:", len(testing_data))
print(
    "Testing period:",
    testing_data["request_date"].min().date(),
    "through",
    testing_data["request_date"].max().date()
)

print()
print("Missing model values:", missing_model_values)

print()
print("MODEL DATA — FIRST FIVE ROWS")

display(
    forecast_model_data[
        [
            "request_date",
            "request_count",
            "trend_index",
            "lag_1",
            "lag_7",
            "lag_14",
            "lag_28",
            "rolling_mean_7",
            "rolling_mean_28",
            "rolling_std_28"
        ]
    ].head()
)

print("MODEL DATA — LAST FIVE ROWS")

display(
    forecast_model_data[
        [
            "request_date",
            "request_count",
            "trend_index",
            "lag_1",
            "lag_7",
            "lag_14",
            "lag_28",
            "rolling_mean_7",
            "rolling_mean_28",
            "rolling_std_28"
        ]
    ].tail()
)

FORECAST DATA PREPARATION
--------------------------------------------------
Complete model rows: 1249
Model features: 32

Training rows: 1159
Training period: 2023-01-29 through 2026-04-01

Testing rows: 90
Testing period: 2026-04-02 through 2026-06-30

Missing model values: 0

MODEL DATA — FIRST FIVE ROWS


,request_date,request_count,trend_index,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_mean_28,rolling_std_28
0,2023-01-29,6998,28,"6,936.00","6,449.00","6,835.00","6,209.00","7,886.29","7,797.82",850.29
1,2023-01-30,8455,29,"6,998.00","7,987.00","7,053.00","6,680.00","7,964.71","7,826.00",807.69
2,2023-01-31,8381,30,"8,455.00","8,851.00","9,287.00","8,255.00","8,031.57","7,889.39",783.72
3,2023-02-01,8687,31,"8,381.00","7,863.00","8,530.00","8,462.00","7,964.43","7,893.89",786.25
4,2023-02-02,9077,32,"8,687.00","8,940.00","7,986.00","8,414.00","8,082.14","7,901.93",793.39


MODEL DATA — LAST FIVE ROWS


,request_date,request_count,trend_index,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_mean_28,rolling_std_28
1244,2026-06-26,11031,1272,"10,955.00","10,239.00","10,998.00","11,629.00","10,915.00","11,172.64",634.09
1245,2026-06-27,9885,1273,"11,031.00","10,414.00","10,184.00","10,085.00","11,028.14","11,151.29",628.20
1246,2026-06-28,10999,1274,"9,885.00","11,685.00","10,698.00","11,464.00","10,952.57","11,144.14",641.76
1247,2026-06-29,11949,1275,"10,999.00","11,305.00","11,535.00","11,810.00","10,854.57","11,127.54",639.19
1248,2026-06-30,11311,1276,"11,949.00","10,440.00","11,242.00","11,593.00","10,946.57","11,132.50",645.20


In [ ]:
# ============================================
# STEP 16: TRAIN AND EVALUATE FORECAST MODEL
# ============================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)

# --------------------------------------------
# 1. PREPARE TRAINING AND TESTING MATRICES
# --------------------------------------------

X_train = training_data[model_feature_columns].astype(float)
y_train = training_data["request_count"].astype(float)

X_test = testing_data[model_feature_columns].astype(float)
y_test = testing_data["request_count"].astype(float)

print("MODEL TRAINING SETUP")
print("-" * 50)

print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))
print("Number of features:", X_train.shape[1])

# --------------------------------------------
# 2. TRAIN LINEAR REGRESSION MODEL
# --------------------------------------------

forecast_model = LinearRegression()

forecast_model.fit(
    X_train,
    y_train
)

# --------------------------------------------
# 3. GENERATE TEST-PERIOD PREDICTIONS
# --------------------------------------------

test_predictions = forecast_model.predict(X_test)

# Prevent impossible negative workload forecasts
test_predictions = np.maximum(
    test_predictions,
    0
)

# --------------------------------------------
# 4. CALCULATE MODEL PERFORMANCE
# --------------------------------------------

model_mae = mean_absolute_error(
    y_test,
    test_predictions
)

model_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        test_predictions
    )
)

model_mape = mean_absolute_percentage_error(
    y_test,
    test_predictions
) * 100

model_r2 = r2_score(
    y_test,
    test_predictions
)

# --------------------------------------------
# 5. CREATE A 7-DAY SEASONAL BASELINE
# --------------------------------------------

baseline_predictions = testing_data["lag_7"].astype(float)

baseline_mae = mean_absolute_error(
    y_test,
    baseline_predictions
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        baseline_predictions
    )
)

baseline_mape = mean_absolute_percentage_error(
    y_test,
    baseline_predictions
) * 100

baseline_r2 = r2_score(
    y_test,
    baseline_predictions
)

# --------------------------------------------
# 6. COMPARE MODEL WITH BASELINE
# --------------------------------------------

performance_comparison = pd.DataFrame({
    "model": [
        "Linear Regression Forecast",
        "7-Day Seasonal Baseline"
    ],
    "mae": [
        model_mae,
        baseline_mae
    ],
    "rmse": [
        model_rmse,
        baseline_rmse
    ],
    "mape_pct": [
        model_mape,
        baseline_mape
    ],
    "r_squared": [
        model_r2,
        baseline_r2
    ]
})

performance_comparison = performance_comparison.round({
    "mae": 2,
    "rmse": 2,
    "mape_pct": 2,
    "r_squared": 4
})

# --------------------------------------------
# 7. BUILD TEST-PERIOD RESULTS TABLE
# --------------------------------------------

forecast_results = testing_data[
    [
        "request_date",
        "request_count",
        "lag_7"
    ]
].copy()

forecast_results = forecast_results.rename(
    columns={
        "request_count": "actual_requests",
        "lag_7": "baseline_forecast"
    }
)

forecast_results["model_forecast"] = np.round(
    test_predictions
).astype(int)

forecast_results["baseline_forecast"] = np.round(
    forecast_results["baseline_forecast"]
).astype(int)

forecast_results["model_error"] = (
    forecast_results["actual_requests"]
    - forecast_results["model_forecast"]
)

forecast_results["absolute_error"] = (
    forecast_results["model_error"]
    .abs()
)

forecast_results["absolute_percentage_error"] = (
    forecast_results["absolute_error"]
    / forecast_results["actual_requests"]
    * 100
).round(2)

# --------------------------------------------
# 8. CALCULATE IMPROVEMENT OVER BASELINE
# --------------------------------------------

mape_improvement_pct = (
    (baseline_mape - model_mape)
    / baseline_mape
    * 100
)

# --------------------------------------------
# 9. DISPLAY RESULTS
# --------------------------------------------

print()
print("MODEL PERFORMANCE COMPARISON")
print("-" * 50)

display(performance_comparison)

print(
    "Model MAPE:",
    f"{model_mape:.2f}%"
)

print(
    "Baseline MAPE:",
    f"{baseline_mape:.2f}%"
)

print(
    "MAPE improvement over baseline:",
    f"{mape_improvement_pct:.2f}%"
)

print()
print("TEST-PERIOD FORECAST RESULTS — FIRST 10 DAYS")

display(
    forecast_results.head(10)
)

print("TEST-PERIOD FORECAST RESULTS — LAST 10 DAYS")

display(
    forecast_results.tail(10)
)

MODEL TRAINING SETUP
--------------------------------------------------
Training observations: 1159
Testing observations: 90
Number of features: 32

MODEL PERFORMANCE COMPARISON
--------------------------------------------------


,model,mae,rmse,mape_pct,r_squared
0,Linear Regression Forecast,514.30,676.70,4.93,0.58
1,7-Day Seasonal Baseline,851.16,"1,107.24",8.23,-0.13


Model MAPE: 4.93%
Baseline MAPE: 8.23%
MAPE improvement over baseline: 40.10%

TEST-PERIOD FORECAST RESULTS — FIRST 10 DAYS


,request_date,actual_requests,baseline_forecast,model_forecast,model_error,absolute_error,absolute_percentage_error
1159,2026-04-02,9650,11429,10788,-1138,1138,11.79
1160,2026-04-03,9652,10700,10170,-518,518,5.37
1161,2026-04-04,8816,9612,9053,-237,237,2.69
1162,2026-04-05,7800,9744,9343,-1543,1543,19.78
1163,2026-04-06,10471,11658,10109,362,362,3.46
1164,2026-04-07,10415,11846,10743,-328,328,3.15
1165,2026-04-08,9966,11076,10261,-295,295,2.96
1166,2026-04-09,9642,9650,10048,-406,406,4.21
1167,2026-04-10,9740,9652,10000,-260,260,2.67
1168,2026-04-11,9219,8816,8957,262,262,2.84


TEST-PERIOD FORECAST RESULTS — LAST 10 DAYS


,request_date,actual_requests,baseline_forecast,model_forecast,model_error,absolute_error,absolute_percentage_error
1239,2026-06-21,11685,10698,10519,1166,1166,9.98
1240,2026-06-22,11305,11535,12399,-1094,1094,9.68
1241,2026-06-23,10440,11242,11613,-1173,1173,11.24
1242,2026-06-24,11367,10409,10733,634,634,5.58
1243,2026-06-25,10955,10816,11268,-313,313,2.86
1244,2026-06-26,11031,10239,11203,-172,172,1.56
1245,2026-06-27,9885,10414,10202,-317,317,3.21
1246,2026-06-28,10999,11685,10374,625,625,5.68
1247,2026-06-29,11949,11305,12127,-178,178,1.49
1248,2026-06-30,11311,10440,11937,-626,626,5.53


In [ ]:
# ============================================
# STEP 17: GENERATE 30-DAY FUTURE FORECAST
# ============================================

# --------------------------------------------
# 1. RETRAIN FINAL MODEL USING ALL MODEL DATA
# --------------------------------------------

X_full = forecast_model_data[
    model_feature_columns
].astype(float)

y_full = forecast_model_data[
    "request_count"
].astype(float)

final_forecast_model = LinearRegression()

final_forecast_model.fit(
    X_full,
    y_full
)

print("Final model trained using:", len(X_full), "observations")

# --------------------------------------------
# 2. PREPARE COMPLETE HISTORICAL SERIES
# --------------------------------------------

historical_series = (
    daily_total[
        [
            "request_date",
            "request_count"
        ]
    ]
    .sort_values("request_date")
    .reset_index(drop=True)
    .copy()
)

history_values = (
    historical_series["request_count"]
    .astype(float)
    .tolist()
)

historical_start_date = historical_series[
    "request_date"
].min()

last_actual_date = historical_series[
    "request_date"
].max()

forecast_horizon = 30

future_dates = pd.date_range(
    start=last_actual_date + pd.Timedelta(days=1),
    periods=forecast_horizon,
    freq="D"
)

# --------------------------------------------
# 3. GENERATE RECURSIVE DAILY FORECASTS
# --------------------------------------------

future_forecast_records = []

for future_date in future_dates:

    trend_index = (
        future_date - historical_start_date
    ).days

    year = future_date.year
    month = future_date.month
    weekday = future_date.dayofweek
    day_of_year = future_date.dayofyear

    is_weekend = int(weekday >= 5)

    annual_sin = np.sin(
        2 * np.pi * day_of_year / 365.25
    )

    annual_cos = np.cos(
        2 * np.pi * day_of_year / 365.25
    )

    lag_1 = history_values[-1]
    lag_7 = history_values[-7]
    lag_14 = history_values[-14]
    lag_28 = history_values[-28]

    rolling_mean_7 = np.mean(
        history_values[-7:]
    )

    rolling_mean_28 = np.mean(
        history_values[-28:]
    )

    rolling_std_28 = pd.Series(
        history_values[-28:]
    ).std(ddof=1)

    # Start every model feature at zero
    future_feature_row = {
        column: 0.0
        for column in model_feature_columns
    }

    # Add numerical features
    future_feature_row.update({
        "trend_index": trend_index,
        "year": year,
        "day_of_year": day_of_year,
        "is_weekend": is_weekend,
        "annual_sin": annual_sin,
        "annual_cos": annual_cos,
        "lag_1": lag_1,
        "lag_7": lag_7,
        "lag_14": lag_14,
        "lag_28": lag_28,
        "rolling_mean_7": rolling_mean_7,
        "rolling_mean_28": rolling_mean_28,
        "rolling_std_28": rolling_std_28
    })

    # Activate the correct weekday indicator
    weekday_column = f"weekday_{weekday}"

    if weekday_column in future_feature_row:
        future_feature_row[weekday_column] = 1.0

    # Activate the correct month indicator
    month_column = f"month_{month}"

    if month_column in future_feature_row:
        future_feature_row[month_column] = 1.0

    future_feature_frame = pd.DataFrame(
        [future_feature_row],
        columns=model_feature_columns
    ).astype(float)

    predicted_requests = final_forecast_model.predict(
        future_feature_frame
    )[0]

    predicted_requests = max(
        predicted_requests,
        0
    )

    predicted_requests = int(
        round(predicted_requests)
    )

    future_forecast_records.append({
        "request_date": future_date,
        "weekday_name": future_date.day_name(),
        "forecast_requests": predicted_requests
    })

    # Add forecast to history so later future dates
    # can use it for lag and rolling features
    history_values.append(
        predicted_requests
    )

# --------------------------------------------
# 4. CREATE FUTURE FORECAST TABLE
# --------------------------------------------

future_forecast = pd.DataFrame(
    future_forecast_records
)

future_forecast["forecast_week"] = (
    future_forecast["request_date"]
    .dt.to_period("W")
    .astype(str)
)

future_forecast["forecast_month"] = (
    future_forecast["request_date"]
    .dt.to_period("M")
    .astype(str)
)

# --------------------------------------------
# 5. DISPLAY FORECAST SUMMARY
# --------------------------------------------

print()
print("30-DAY FUTURE WORKLOAD FORECAST")
print("-" * 50)

print(
    "Forecast period:",
    future_forecast["request_date"].min().date(),
    "through",
    future_forecast["request_date"].max().date()
)

print(
    "Total forecast requests:",
    f"{future_forecast['forecast_requests'].sum():,.0f}"
)

print(
    "Average daily forecast:",
    f"{future_forecast['forecast_requests'].mean():,.2f}"
)

print(
    "Highest daily forecast:",
    f"{future_forecast['forecast_requests'].max():,.0f}"
)

print(
    "Lowest daily forecast:",
    f"{future_forecast['forecast_requests'].min():,.0f}"
)

print()
display(future_forecast)

# --------------------------------------------
# 6. SAVE FORECAST FOR POWER BI
# --------------------------------------------

future_forecast_file = (
    OUTPUT_FOLDER
    / "future_30_day_forecast.csv"
)

future_forecast.to_csv(
    future_forecast_file,
    index=False
)

print()
print("Forecast file saved successfully:")
print(future_forecast_file)

Final model trained using: 1249 observations

30-DAY FUTURE WORKLOAD FORECAST
--------------------------------------------------
Forecast period: 2026-07-01 through 2026-07-30
Total forecast requests: 330,941
Average daily forecast: 11,031.37
Highest daily forecast: 11,819
Lowest daily forecast: 9,969



,request_date,weekday_name,forecast_requests,forecast_week,forecast_month
0,2026-07-01,Wednesday,10992,2026-06-29/2026-07-05,2026-07
1,2026-07-02,Thursday,10876,2026-06-29/2026-07-05,2026-07
2,2026-07-03,Friday,10937,2026-06-29/2026-07-05,2026-07
3,2026-07-04,Saturday,9969,2026-06-29/2026-07-05,2026-07
4,2026-07-05,Sunday,10336,2026-06-29/2026-07-05,2026-07
5,2026-07-06,Monday,11597,2026-07-06/2026-07-12,2026-07
6,2026-07-07,Tuesday,11531,2026-07-06/2026-07-12,2026-07
7,2026-07-08,Wednesday,11159,2026-07-06/2026-07-12,2026-07
8,2026-07-09,Thursday,10996,2026-07-06/2026-07-12,2026-07
9,2026-07-10,Friday,11040,2026-07-06/2026-07-12,2026-07



Forecast file saved successfully:
/content/project_2_demand_forecasting/outputs/future_30_day_forecast.csv


In [ ]:
# ============================================
# STEP 18: CREATE CAPACITY-RISK ALERTS
# ============================================

# --------------------------------------------
# 1. CALCULATE DATA-DRIVEN DAILY CAPACITY
# --------------------------------------------

recent_capacity_window = (
    daily_total
    .sort_values("request_date")
    .tail(180)
    .copy()
)

planning_capacity = int(
    round(
        recent_capacity_window[
            "request_count"
        ].quantile(0.75)
    )
)

recent_average = recent_capacity_window[
    "request_count"
].mean()

recent_peak = recent_capacity_window[
    "request_count"
].max()

print("CAPACITY PLANNING ASSUMPTIONS")
print("-" * 50)

print(
    "Historical capacity window:",
    recent_capacity_window["request_date"].min().date(),
    "through",
    recent_capacity_window["request_date"].max().date()
)

print(
    "Recent average daily workload:",
    f"{recent_average:,.2f}"
)

print(
    "Planning capacity:",
    f"{planning_capacity:,.0f}",
    "requests per day"
)

print(
    "Recent peak workload:",
    f"{recent_peak:,.0f}"
)

# --------------------------------------------
# 2. CREATE CAPACITY ALERT TABLE
# --------------------------------------------

capacity_alerts = future_forecast.copy()

capacity_alerts["days_ahead"] = (
    capacity_alerts["request_date"]
    - last_actual_date
).dt.days

capacity_alerts["planning_capacity"] = (
    planning_capacity
)

capacity_alerts["capacity_utilization_pct"] = (
    capacity_alerts["forecast_requests"]
    / capacity_alerts["planning_capacity"]
    * 100
).round(2)

capacity_alerts["capacity_gap_requests"] = (
    capacity_alerts["forecast_requests"]
    - capacity_alerts["planning_capacity"]
).clip(lower=0)

capacity_alerts["available_capacity"] = (
    capacity_alerts["planning_capacity"]
    - capacity_alerts["forecast_requests"]
).clip(lower=0)

# --------------------------------------------
# 3. ASSIGN RISK LEVELS
# --------------------------------------------

def assign_capacity_risk(utilization):
    if utilization >= 110:
        return "Critical"
    elif utilization > 100:
        return "High"
    elif utilization >= 95:
        return "Medium"
    else:
        return "Low"


capacity_alerts["risk_level"] = (
    capacity_alerts["capacity_utilization_pct"]
    .apply(assign_capacity_risk)
)

capacity_alerts["capacity_shortfall_flag"] = np.where(
    capacity_alerts["capacity_gap_requests"] > 0,
    1,
    0
)

# --------------------------------------------
# 4. ADD BUSINESS RECOMMENDATIONS
# --------------------------------------------

def recommended_action(row):
    if row["risk_level"] == "Critical":
        return "Activate surge capacity and escalate immediately"
    elif row["risk_level"] == "High":
        return "Add capacity and prioritize high-impact requests"
    elif row["risk_level"] == "Medium":
        return "Monitor closely and prepare backup capacity"
    else:
        return "Normal operating plan"


capacity_alerts["recommended_action"] = (
    capacity_alerts.apply(
        recommended_action,
        axis=1
    )
)

# --------------------------------------------
# 5. CALCULATE RISK SUMMARY
# --------------------------------------------

risk_summary = (
    capacity_alerts
    .groupby(
        "risk_level",
        as_index=False
    )
    .agg(
        forecast_days=("request_date", "count"),
        total_forecast_requests=(
            "forecast_requests",
            "sum"
        ),
        average_utilization_pct=(
            "capacity_utilization_pct",
            "mean"
        ),
        total_capacity_gap=(
            "capacity_gap_requests",
            "sum"
        )
    )
)

risk_order = {
    "Critical": 1,
    "High": 2,
    "Medium": 3,
    "Low": 4
}

risk_summary["risk_order"] = (
    risk_summary["risk_level"]
    .map(risk_order)
)

risk_summary = (
    risk_summary
    .sort_values("risk_order")
    .drop(columns="risk_order")
    .reset_index(drop=True)
)

risk_summary["average_utilization_pct"] = (
    risk_summary["average_utilization_pct"]
    .round(2)
)

# --------------------------------------------
# 6. IDENTIFY FIRST CAPACITY SHORTFALL
# --------------------------------------------

shortfall_days = capacity_alerts[
    capacity_alerts["capacity_shortfall_flag"] == 1
].copy()

print()
print("30-DAY CAPACITY-RISK SUMMARY")
print("-" * 50)

print(
    "Forecast days:",
    len(capacity_alerts)
)

print(
    "Capacity-shortfall days:",
    int(
        capacity_alerts[
            "capacity_shortfall_flag"
        ].sum()
    )
)

print(
    "Total forecast capacity gap:",
    f"{capacity_alerts['capacity_gap_requests'].sum():,.0f}",
    "requests"
)

print(
    "Maximum forecast utilization:",
    f"{capacity_alerts['capacity_utilization_pct'].max():.2f}%"
)

if len(shortfall_days) > 0:

    first_shortfall = shortfall_days.iloc[0]

    print()
    print(
        "First predicted capacity shortfall:",
        first_shortfall["request_date"].date()
    )

    print(
        "Advance warning:",
        int(first_shortfall["days_ahead"]),
        "days ahead"
    )

    print(
        "Expected shortfall:",
        f"{first_shortfall['capacity_gap_requests']:,.0f}",
        "requests"
    )

else:
    print()
    print(
        "No capacity shortfall is predicted "
        "during the 30-day forecast."
    )

print()
print("RISK-LEVEL SUMMARY")

display(risk_summary)

print("DAILY CAPACITY ALERTS")

display(
    capacity_alerts[
        [
            "request_date",
            "weekday_name",
            "days_ahead",
            "forecast_requests",
            "planning_capacity",
            "capacity_utilization_pct",
            "capacity_gap_requests",
            "risk_level",
            "recommended_action"
        ]
    ]
)

# --------------------------------------------
# 7. SAVE FILES FOR POWER BI
# --------------------------------------------

capacity_alerts_file = (
    OUTPUT_FOLDER
    / "capacity_risk_alerts.csv"
)

risk_summary_file = (
    OUTPUT_FOLDER
    / "capacity_risk_summary.csv"
)

capacity_alerts.to_csv(
    capacity_alerts_file,
    index=False
)

risk_summary.to_csv(
    risk_summary_file,
    index=False
)

print()
print("Capacity-alert files saved successfully:")
print(capacity_alerts_file)
print(risk_summary_file)

CAPACITY PLANNING ASSUMPTIONS
--------------------------------------------------
Historical capacity window: 2026-01-02 through 2026-06-30
Recent average daily workload: 11,034.96
Planning capacity: 11,609 requests per day
Recent peak workload: 22,806

30-DAY CAPACITY-RISK SUMMARY
--------------------------------------------------
Forecast days: 30
Capacity-shortfall days: 5
Total forecast capacity gap: 566 requests
Maximum forecast utilization: 101.81%

First predicted capacity shortfall: 2026-07-13
Advance warning: 13 days ahead
Expected shortfall: 12 requests

RISK-LEVEL SUMMARY


,risk_level,forecast_days,total_forecast_requests,average_utilization_pct,total_capacity_gap
0,High,5,58611,100.97,566
1,Medium,13,146731,97.23,0
2,Low,12,125599,90.16,0


DAILY CAPACITY ALERTS


,request_date,weekday_name,days_ahead,forecast_requests,planning_capacity,capacity_utilization_pct,capacity_gap_requests,risk_level,recommended_action
0,2026-07-01,Wednesday,1,10992,11609,94.69,0,Low,Normal operating plan
1,2026-07-02,Thursday,2,10876,11609,93.69,0,Low,Normal operating plan
2,2026-07-03,Friday,3,10937,11609,94.21,0,Low,Normal operating plan
3,2026-07-04,Saturday,4,9969,11609,85.87,0,Low,Normal operating plan
4,2026-07-05,Sunday,5,10336,11609,89.03,0,Low,Normal operating plan
5,2026-07-06,Monday,6,11597,11609,99.90,0,Medium,Monitor closely and prepare backup capacity
6,2026-07-07,Tuesday,7,11531,11609,99.33,0,Medium,Monitor closely and prepare backup capacity
7,2026-07-08,Wednesday,8,11159,11609,96.12,0,Medium,Monitor closely and prepare backup capacity
8,2026-07-09,Thursday,9,10996,11609,94.72,0,Low,Normal operating plan
9,2026-07-10,Friday,10,11040,11609,95.10,0,Medium,Monitor closely and prepare backup capacity



Capacity-alert files saved successfully:
/content/project_2_demand_forecasting/outputs/capacity_risk_alerts.csv
/content/project_2_demand_forecasting/outputs/capacity_risk_summary.csv


In [ ]:
# ============================================
# STEP 19: CREATE POWER BI DATA PACKAGE
# ============================================

import shutil

POWER_BI_FOLDER = OUTPUT_FOLDER / "power_bi_data"

POWER_BI_FOLDER.mkdir(
    parents=True,
    exist_ok=True
)

# --------------------------------------------
# 1. CREATE POWER BI DATE TABLE
# --------------------------------------------

calendar_start = daily_total["request_date"].min()
calendar_end = future_forecast["request_date"].max()

date_dimension = pd.DataFrame({
    "date": pd.date_range(
        start=calendar_start,
        end=calendar_end,
        freq="D"
    )
})

date_dimension["year"] = date_dimension["date"].dt.year
date_dimension["quarter"] = (
    "Q" + date_dimension["date"].dt.quarter.astype(str)
)
date_dimension["month_number"] = date_dimension["date"].dt.month
date_dimension["month_name"] = date_dimension["date"].dt.month_name()
date_dimension["month_short"] = (
    date_dimension["date"].dt.strftime("%b")
)
date_dimension["year_month"] = (
    date_dimension["date"].dt.strftime("%Y-%m")
)
date_dimension["year_month_sort"] = (
    date_dimension["date"].dt.strftime("%Y%m").astype(int)
)
date_dimension["day_name"] = date_dimension["date"].dt.day_name()
date_dimension["weekday_number"] = (
    date_dimension["date"].dt.dayofweek + 1
)
date_dimension["day_of_month"] = date_dimension["date"].dt.day
date_dimension["week_of_year"] = (
    date_dimension["date"]
    .dt.isocalendar()
    .week
    .astype(int)
)
date_dimension["week_start"] = (
    date_dimension["date"]
    - pd.to_timedelta(
        date_dimension["date"].dt.dayofweek,
        unit="D"
    )
)
date_dimension["is_weekend"] = (
    date_dimension["date"].dt.dayofweek >= 5
).astype(int)

# --------------------------------------------
# 2. CREATE ACTUAL + FORECAST TIME SERIES
# --------------------------------------------

historical_power_bi = daily_total.copy()

historical_power_bi = historical_power_bi.rename(
    columns={
        "request_date": "date",
        "request_count": "actual_requests"
    }
)

historical_power_bi["forecast_requests"] = np.nan
historical_power_bi["record_type"] = "Actual"

future_power_bi = future_forecast[
    [
        "request_date",
        "forecast_requests"
    ]
].copy()

future_power_bi = future_power_bi.rename(
    columns={
        "request_date": "date"
    }
)

future_power_bi["actual_requests"] = np.nan
future_power_bi["record_type"] = "Forecast"

actual_forecast_series = pd.concat(
    [
        historical_power_bi[
            [
                "date",
                "actual_requests",
                "forecast_requests",
                "record_type"
            ]
        ],
        future_power_bi[
            [
                "date",
                "actual_requests",
                "forecast_requests",
                "record_type"
            ]
        ]
    ],
    ignore_index=True
)

actual_forecast_series = (
    actual_forecast_series
    .sort_values("date")
    .reset_index(drop=True)
)

# --------------------------------------------
# 3. CREATE MODEL METRICS TABLE
# --------------------------------------------

model_metrics = pd.DataFrame({
    "model_name": [
        "Linear Regression Forecast",
        "7-Day Seasonal Baseline"
    ],
    "mae": [
        model_mae,
        baseline_mae
    ],
    "rmse": [
        model_rmse,
        baseline_rmse
    ],
    "mape_pct": [
        model_mape,
        baseline_mape
    ],
    "r_squared": [
        model_r2,
        baseline_r2
    ]
})

model_metrics["mape_improvement_pct"] = [
    mape_improvement_pct,
    0
]

model_metrics = model_metrics.round({
    "mae": 2,
    "rmse": 2,
    "mape_pct": 2,
    "r_squared": 4,
    "mape_improvement_pct": 2
})

# --------------------------------------------
# 4. CREATE PROJECT KPI TABLE
# --------------------------------------------

ytd_2026_growth = ytd_workload.loc[
    ytd_workload["request_year"] == 2026,
    "ytd_growth_pct"
].iloc[0]

if len(shortfall_days) > 0:
    first_shortfall_date = (
        shortfall_days.iloc[0]["request_date"]
        .strftime("%Y-%m-%d")
    )

    first_shortfall_warning_days = int(
        shortfall_days.iloc[0]["days_ahead"]
    )
else:
    first_shortfall_date = None
    first_shortfall_warning_days = None

project_kpis = pd.DataFrame({
    "historical_total_requests": [
        int(daily_total["request_count"].sum())
    ],
    "historical_average_daily_requests": [
        round(daily_total["request_count"].mean(), 2)
    ],
    "ytd_2026_growth_pct": [
        round(ytd_2026_growth, 2)
    ],
    "model_mape_pct": [
        round(model_mape, 2)
    ],
    "baseline_mape_pct": [
        round(baseline_mape, 2)
    ],
    "mape_improvement_pct": [
        round(mape_improvement_pct, 2)
    ],
    "forecast_30_day_requests": [
        int(future_forecast["forecast_requests"].sum())
    ],
    "forecast_average_daily_requests": [
        round(
            future_forecast["forecast_requests"].mean(),
            2
        )
    ],
    "planning_capacity_threshold": [
        planning_capacity
    ],
    "capacity_shortfall_days": [
        int(
            capacity_alerts[
                "capacity_shortfall_flag"
            ].sum()
        )
    ],
    "total_forecast_capacity_gap": [
        int(
            capacity_alerts[
                "capacity_gap_requests"
            ].sum()
        )
    ],
    "maximum_forecast_utilization_pct": [
        round(
            capacity_alerts[
                "capacity_utilization_pct"
            ].max(),
            2
        )
    ],
    "first_shortfall_date": [
        first_shortfall_date
    ],
    "advance_warning_days": [
        first_shortfall_warning_days
    ]
})

# --------------------------------------------
# 5. PREPARE DATE COLUMNS FOR CSV EXPORT
# --------------------------------------------

daily_total_export = daily_total.copy()
daily_segments_export = daily_segments.copy()
monthly_workload_export = monthly_workload.copy()
forecast_results_export = forecast_results.copy()
future_forecast_export = future_forecast.copy()
capacity_alerts_export = capacity_alerts.copy()

daily_total_export["request_date"] = (
    daily_total_export["request_date"]
    .dt.strftime("%Y-%m-%d")
)

daily_segments_export["request_date"] = (
    daily_segments_export["request_date"]
    .dt.strftime("%Y-%m-%d")
)

monthly_workload_export["month_start"] = (
    monthly_workload_export["month_start"]
    .dt.strftime("%Y-%m-%d")
)

forecast_results_export["request_date"] = (
    forecast_results_export["request_date"]
    .dt.strftime("%Y-%m-%d")
)

future_forecast_export["request_date"] = (
    future_forecast_export["request_date"]
    .dt.strftime("%Y-%m-%d")
)

capacity_alerts_export["request_date"] = (
    capacity_alerts_export["request_date"]
    .dt.strftime("%Y-%m-%d")
)

date_dimension_export = date_dimension.copy()

date_dimension_export["date"] = (
    date_dimension_export["date"]
    .dt.strftime("%Y-%m-%d")
)

date_dimension_export["week_start"] = (
    date_dimension_export["week_start"]
    .dt.strftime("%Y-%m-%d")
)

actual_forecast_export = actual_forecast_series.copy()

actual_forecast_export["date"] = (
    actual_forecast_export["date"]
    .dt.strftime("%Y-%m-%d")
)

# --------------------------------------------
# 6. SAVE ALL POWER BI TABLES
# --------------------------------------------

power_bi_tables = {
    "date_dimension.csv": date_dimension_export,
    "daily_total.csv": daily_total_export,
    "daily_segments.csv": daily_segments_export,
    "yearly_workload.csv": yearly_workload,
    "ytd_workload.csv": ytd_workload,
    "monthly_workload.csv": monthly_workload_export,
    "borough_workload.csv": borough_workload,
    "agency_workload.csv": agency_workload,
    "weekday_workload.csv": weekday_workload,
    "model_metrics.csv": model_metrics,
    "forecast_test_results.csv": forecast_results_export,
    "future_30_day_forecast.csv": future_forecast_export,
    "capacity_risk_alerts.csv": capacity_alerts_export,
    "capacity_risk_summary.csv": risk_summary,
    "actual_forecast_series.csv": actual_forecast_export,
    "project_kpis.csv": project_kpis
}

for filename, dataframe in power_bi_tables.items():

    file_path = POWER_BI_FOLDER / filename

    dataframe.to_csv(
        file_path,
        index=False
    )

# --------------------------------------------
# 7. CREATE ZIP PACKAGE
# --------------------------------------------

zip_base_path = (
    PROJECT_FOLDER
    / "Project_2_PowerBI_Data"
)

zip_file_path = shutil.make_archive(
    str(zip_base_path),
    "zip",
    root_dir=POWER_BI_FOLDER
)

# --------------------------------------------
# 8. DISPLAY EXPORT SUMMARY
# --------------------------------------------

export_summary = pd.DataFrame({
    "file_name": list(power_bi_tables.keys()),
    "rows": [
        len(dataframe)
        for dataframe in power_bi_tables.values()
    ]
})

print("POWER BI DATA PACKAGE CREATED")
print("-" * 50)

print(
    "Power BI tables:",
    len(power_bi_tables)
)

print(
    "Folder:",
    POWER_BI_FOLDER
)

print(
    "ZIP file:",
    zip_file_path
)

print()
display(export_summary)

print()
print("PROJECT KPI SUMMARY")
display(project_kpis)

POWER BI DATA PACKAGE CREATED
--------------------------------------------------
Power BI tables: 16
Folder: /content/project_2_demand_forecasting/outputs/power_bi_data
ZIP file: /content/project_2_demand_forecasting/Project_2_PowerBI_Data.zip



,file_name,rows
0,date_dimension.csv,1307
1,daily_total.csv,1277
2,daily_segments.csv,76458
3,yearly_workload.csv,4
4,ytd_workload.csv,2
5,monthly_workload.csv,42
6,borough_workload.csv,5
7,agency_workload.csv,17
8,weekday_workload.csv,7
9,model_metrics.csv,2



PROJECT KPI SUMMARY


,historical_total_requests,historical_average_daily_requests,ytd_2026_growth_pct,model_mape_pct,baseline_mape_pct,mape_improvement_pct,forecast_30_day_requests,forecast_average_daily_requests,planning_capacity_threshold,capacity_shortfall_days,total_forecast_capacity_gap,maximum_forecast_utilization_pct,first_shortfall_date,advance_warning_days
0,12331111,"9,656.31",13.40,4.93,8.23,40.10,330941,"11,031.37",11609,5,566,101.81,2026-07-13,13


In [ ]:
# ============================================
# STEP 20: DOWNLOAD POWER BI DATA PACKAGE
# ============================================

from google.colab import files
from pathlib import Path

download_file = Path(
    "/content/project_2_demand_forecasting/"
    "Project_2_PowerBI_Data.zip"
)

if not download_file.exists():
    raise FileNotFoundError(
        f"ZIP file was not found: {download_file}"
    )

file_size_mb = download_file.stat().st_size / (1024 * 1024)

print("File ready for download:")
print(download_file)
print(f"File size: {file_size_mb:.2f} MB")

files.download(str(download_file))

File ready for download:
/content/project_2_demand_forecasting/Project_2_PowerBI_Data.zip
File size: 0.34 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>